# Introduction

In this project, my goal is to build a deep learning model--specifically, an LSTM on top of the BERTweet model--to predict whether a given tweet is about a real disaster (class 1) or not(class 0).

**Data Source:** Addison Howard, devrishi, Phil Culliton, and Yufeng Guo. Natural Language Processing with Disaster Tweets. https://kaggle.com/competitions/nlp-getting-started, 2019. Kaggle.


In [ ]:
# Install emoji as required by BERTWeet
!pip install emoji==0.6.0

# Basic packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import os
import random

# Modeling
import torch
from torch import nn
from torch.utils.data import DataLoader
import transformers
from transformers import AutoModel, AutoTokenizer
import sklearn
from sklearn.model_selection import train_test_split
from transformers import set_seed as hf_set_seed

# Set environment variable to solve non-determinism issue for RNN functions.
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8" 

# Set global seed and deterministic behavior
def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)   
    hf_set_seed(42)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [ ]:
# Check the version of Transformers and PyTorch
print(f"Transformers version: {transformers.__version__}")
print(f"Torch version: {torch.__version__}")

# Setup device agnostic code
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

In [ ]:
# Load the data
train = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')
test = pd.read_csv('/kaggle/input/nlp-getting-started/test.csv')

# Data Exploring and Cleaning

In [ ]:
# Take a look at the data
print(train.info())
train.head()

The training set contains 7613 records with five attributes: `id`, `keyword`, `location`, `text`, and `target`, where the `text` column contains tweets and the `target` column provides the label of the tweet (1: disaster, 0: non-disaster). As we can see from the dataset info, many null value exist in the `location` and `keyword` columns. Let's take a closer look at these two columns.

In [ ]:
# Examine values of 'keyword' and 'location'
uni_keyword = len(train.keyword.unique())
null_keyword = train.keyword.isnull().sum()/train.shape[0]
uni_location = len(train.location.unique())
null_location = train.location.isnull().sum()/train.shape[0]

print(f"Keyword: {null_keyword*100:.1f}% null values, {uni_keyword} unique values")
print(f"Location: {null_location*100:.1f}% null values, {uni_location} unique values")

The percentage of null value in the `keyword` column is less than 1%, which is acceptable. However, the `location` column contains 33.3% null values and 3,342 unique values out of 5,080 non-null entries, suggesting that it doesn't provide much useful information. Therefore, I will remove `location` column and clean the `keyword` column.

In [ ]:
# Remove location
clean_train = train.drop(columns=['location'])
clean_test = test.drop(columns=['location'])

In [ ]:
# Take a look at 'keyword' values
clean_train.keyword.value_counts()

In [ ]:
# Clean keywords
def clean_keyword(keyword):
    keyword.fillna('', inplace=True)   # replace NaN with ''
    keyword = keyword.str.replace('%20', ' ')   # replace '%20' with a space
    keyword = keyword.str.replace(r'\s+', ' ', regex=True) # remove extra space
    return keyword

clean_train['keyword'] = clean_keyword(clean_train['keyword'])
clean_test['keyword'] = clean_keyword(clean_test['keyword'])

In [ ]:
# Take a look at the text data
pd.set_option('display.max_colwidth', 200)

for label, group in clean_train.groupby('target'):
    print(f"Label: {label}")
    print(group['text'].str.ljust(200).sample(5))
    print('')

The Bertweet tokenizer would perform the following text normalization and tokenization for us--

1. Tokenize tweets using `TweetTokenizer` from the `NLTK` toolkit;
2. Translate emotion icons into text strings using `emoji` package;
3. Convert user mentions and web/url links into special tokens @USER and HTTPURL.

However, since I need to remove duplicate texts--especially those with different labels to avoid confusion--I will define a function to do some minimal text preprocessing.

In [ ]:
# Define a function for minimal cleaning
def minimal_cleaning(text):
    text = text.replace('\n', ' ')            # replace \n with a space
    text = text.replace('w/', ' with ')       # replace w/ with 'with'
    text = text.replace('&amp', ' and ')      # replace &amp with 'and'
    text = text.replace('Ûªm', ' am ')        # replace Ûªm with 'am'
    text = re.sub(r'@\S+', '@USER', text)     # replace mentions with @USER
    text = re.sub(r'http\S+', 'HTTPURL', text) #replace url with 'HTTPURL'
    text = re.sub(r'#\S+', '#TOPIC', text)    # replace hashtags with #TOPIC
    text = re.sub(r'\s+', ' ', text)          # remove extra space

    return text

In [ ]:
# Add new columns of 'cleaned_text' to clean_train and clean_test dataframe
clean_train['cleaned_text'] = clean_train['text'].apply(minimal_cleaning)
clean_test['cleaned_text'] = clean_test['text'].apply(minimal_cleaning)

In [ ]:
# Check duplicated text with the same labels
clean_train[clean_train.duplicated(subset=['cleaned_text', 'target'])].shape[0]

In [ ]:
# Drop duplicated text with the same target labels, keep only the first occurrence
clean_train = clean_train.drop_duplicates(subset = ['cleaned_text', 'target'])

In [ ]:
# Check the number of duplicated text with different target labels
clean_train[clean_train.duplicated(subset = 'cleaned_text', keep = False)].shape[0]

There are 144 duplicated text with different target labels. I will remove them to avoid confusion while modeling.

In [ ]:
# Drop duplicated text with the different target labels
clean_train = clean_train.drop_duplicates(subset = ['cleaned_text'], keep = False)

In [ ]:
# Concatenate 'keyword' with 'cleaned_text'
clean_train['cleaned_text'] = clean_train['keyword'].str.cat(others=clean_train['cleaned_text'], sep='. ')
clean_test['cleaned_text'] = clean_test['keyword'].str.cat(others=clean_test['cleaned_text'], sep='. ')

# Remove the '. ' in the beginning of a string(rows with no keyword)
clean_train['cleaned_text'] = clean_train['cleaned_text'].str.replace(r'^\. ', '', regex=True)
clean_test['cleaned_text'] = clean_test['cleaned_text'].str.replace(r'^\. ', '', regex=True)

In [ ]:
# Drop 'keyword' and 'text', and reset index
clean_train = clean_train.drop(columns=['keyword', 'text']).reset_index(drop=True)
clean_test = clean_test.drop(columns=['keyword', 'text']).reset_index(drop=True)

In [ ]:
clean_train.info()

After cleaning, there are 6868 entries remaining in our training set. Let's check the distribution of the target labels.

In [ ]:
# Visualize the target label
labels = clean_train['target'].value_counts().index
values = clean_train['target'].value_counts().values

plt.pie(values, labels=labels, colors=['skyblue', 'lightsalmon'], autopct='%1.0f%%')
plt.title('The Distribution of Targets');

# Tokenization and DataLoader Setup

In [ ]:
# Load the tokenizer
bertweet_tokenizer = AutoTokenizer.from_pretrained("vinai/bertweet-base", normalization=True, use_fast=False)

In [ ]:
# Examine the text length
text_length = clean_train['cleaned_text'].str.split().str.len()
text_length.describe()

In [ ]:
longest_line = clean_train[clean_train['cleaned_text'].str.split().str.len()==33]['cleaned_text'].to_string(index=False)
bertweet_tokenizer(longest_line, return_tensors='pt')['input_ids'].shape

The longest text has a length of 39 tokens. I will set the max sequence length to be 42.

In [ ]:
# Split the training data into a training set and a validation set
training_set, validation_set = train_test_split(clean_train, test_size=0.2, stratify=clean_train['target'], random_state=42)
len(training_set), len(validation_set)

In [ ]:
# Tokenize text data
train_tokenized = bertweet_tokenizer(training_set['cleaned_text'].to_list(),
                                     padding=True,
                                     truncation=True,
                                     max_length=42,
                                     return_tensors='pt')

val_tokenized = bertweet_tokenizer(validation_set['cleaned_text'].to_list(),
                                    padding=True,
                                    truncation=True,
                                    max_length=42,
                                    return_tensors='pt')

test_tokenized = bertweet_tokenizer(clean_test['cleaned_text'].to_list(),
                                    padding=True,
                                    truncation=True,
                                    max_length=42,
                                    return_tensors='pt')

In [ ]:
# Create torch dataset
class tweet_Dataset(torch.utils.data.Dataset):
    def __init__(self, tokenized, labels=None):
        self.tokenized = tokenized
        self.labels = labels

    def __len__(self):
        return len(self.tokenized['input_ids'])

    def __getitem__(self, idx):
        # Rename input_ids to inputs
        item = {'input_ids': self.tokenized['input_ids'][idx],
                'attention_mask': self.tokenized['attention_mask'][idx],
                'token_type_ids': self.tokenized['token_type_ids'][idx]}

        if self.labels is not None: # Check if labels exist before adding
            item['labels'] = torch.tensor(self.labels[idx])

        return item

In [ ]:
train_dataset = tweet_Dataset(train_tokenized, training_set['target'].to_list())
val_dataset = tweet_Dataset(val_tokenized, validation_set['target'].to_list())
test_dataset = tweet_Dataset(test_tokenized)

train_dataset[0]

In [ ]:
# Set seed for dataloader
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(42)

In [ ]:
# Create data loader
train_dataloader = DataLoader(train_dataset, batch_size=8, shuffle=True, 
                              worker_init_fn=seed_worker, generator=g)
val_dataloader = DataLoader(val_dataset, batch_size=8, shuffle=True,
                           worker_init_fn=seed_worker, generator=g)
test_dataloader = DataLoader(test_dataset, batch_size=8, shuffle=False,
                            worker_init_fn=seed_worker, generator=g)

# Modeling

In [ ]:
bertweet_model = AutoModel.from_pretrained("vinai/bertweet-base")

In [ ]:
# Send model to 'cuda' if available
bertweet_model.to(device)

In [ ]:
def trainable_parameters(model):
    return sum([p.numel() for p in model.parameters() if p.requires_grad])

print(f"There are {trainable_parameters(bertweet_model)} trainable parameters in the model.")

As describe in the paper, BERTweet model uses the same architecture as BERT_base and is trained using the RoBERTa pretraining procedure.

Since the model was pretrained on 850 million Tweets (16 billion word tokens), and our dataset is extremely small in comparison, I will freeze all the parameters in the model and simply add a classification layer on top of the pooled output as our first model.

In [ ]:
# Freeze bertweet model parameters
for name, param in bertweet_model.named_parameters():
    param.requires_grad = False

print(f"There are {trainable_parameters(bertweet_model)} trainable parameters in the model.")

In [ ]:
# Create the BERTWeet_Classification model by subclassing nn.Module
class BERTWeet_Classification(nn.Module):
    def __init__(self, pretrained_model, dropout=0.3):
        super().__init__()
        self.bertweet = pretrained_model
        self.dropout = nn.Dropout(p=dropout)
        self.classifier = nn.Linear(768, 2)
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None):
        outputs =  self.bertweet(input_ids=input_ids, 
                                 attention_mask=attention_mask, 
                                 token_type_ids=token_type_ids)
        x = self.dropout(outputs.pooler_output)
        logits = self.classifier(x)
        return logits

I'll define a few helper functions--train steps, evaluation steps and checkpointing--for the `train_epoch` function.

In [ ]:
# Define train step function
def train_step(model, dataloader, optimizer, loss_fn, device):
    # Set model to training mode
    model.train()

    # Setup train loss value
    train_loss, train_accu = 0, 0

    # Loop through dataloader
    for i, data in enumerate(dataloader):
        input_ids = data['input_ids'].to(device)
        attention_mask = data['attention_mask'].to(device)
        token_type_ids = data['token_type_ids'].to(device)
        labels = data['labels'].to(device)
        
        # Zero the grad
        optimizer.zero_grad()

        # Forward + backward + gradient step
        logits = model(input_ids, attention_mask, token_type_ids)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()

        # loss / accu
        train_loss += loss.item()
        y_preds = torch.softmax(logits, dim=1).argmax(dim=1)
        accu = (y_preds == labels).sum().item()/len(y_preds)
        train_accu += accu
        
    # Get average loss/accu
    train_loss /= len(dataloader)
    train_accu /= len(dataloader)
    
    return train_loss, train_accu

# Define evaluation step function
def eval_step(model, dataloader, loss_fn, device):
    # Set model to evaluation mode
    model.eval()

    # Setup eval_loss and eval_accu values
    eval_loss, eval_accu = 0, 0

    # Turn on inference context manager + Loop through dataloader
    with torch.inference_mode():
        for i, data in enumerate(dataloader):
            input_ids = data['input_ids'].to(device)
            attention_mask = data['attention_mask'].to(device)
            token_type_ids = data['token_type_ids'].to(device)
            labels = data['labels'].to(device)
            
            # Forward + calculate loss/accu
            logits = model(input_ids, attention_mask, token_type_ids)
            eval_loss += loss_fn(logits, labels).item()
            y_preds = torch.softmax(logits, dim=1).argmax(dim=1)
            accu = (y_preds == labels).sum().item()/len(y_preds)
            eval_accu += accu
            
    # Get average loss/accu
    eval_loss /= len(dataloader)
    eval_accu /= len(dataloader)

    return eval_loss, eval_accu         

# Define checkpoint function
def checkpoint(model, filename):
    torch.save(model.state_dict(), filename)

def resume_checkpoint(model, filename):
    model.load_state_dict(torch.load(filename))

In [ ]:
from tqdm.auto import tqdm

# Define train_epoch function with early stopping and checkpoint
def train_epoch(model, train_dataloader, eval_dataloader, 
                optimizer, loss_fn, early_stop_patience, 
                checkpoint_file, device, epochs=10):

    # Create the results dictionary
    results = {
        "train_loss": [],
        'train_accu': [],
        "eval_loss": [],
        "eval_accu": []
    }

    # Setup best_epoch and best_accuracy values
    best_epoch = -1
    best_accu = -1
    
    # Loop through epochs
    for epoch in tqdm(range(1, epochs+1)):
        # Train step
        train_loss, train_accu = train_step(model=model, 
                                dataloader=train_dataloader, 
                                optimizer=optimizer, 
                                loss_fn=loss_fn, 
                                device=device)
        
        # Evaluation step
        eval_loss, eval_accu = eval_step(model=model, 
                                         dataloader=eval_dataloader, 
                                         loss_fn=loss_fn, 
                                         device=device)
        # Print out what's happening
        print(f"Epoch: {epoch} |"
              f"train_loss: {train_loss:.4f} |"
              f"train_accu: {train_accu:.4f} |"
              f"eval_loss: {eval_loss:.4f} |"
              f"eval_accu: {eval_accu:.4f}")

        # Update the results dictionary
        results['train_loss'].append(train_loss)
        results['train_accu'].append(train_accu)
        results['eval_loss'].append(eval_loss)
        results['eval_accu'].append(eval_accu)

        # Checkpoint + Early Stopping
        if eval_accu > best_accu:
            best_accu = eval_accu
            best_epoch = epoch
            checkpoint(model, checkpoint_file)
        elif epoch - best_epoch >= early_stop_patience:
            print(f"Early stopped at epoch {epoch}."
                  f"Best eval_accu {best_accu} at epoch {best_epoch}.")
            resume_checkpoint(model, checkpoint_file)
            return results

    # Save the best model in the end
    resume_checkpoint(model, checkpoint_file)
    print(f"Finished training."
          f"Best eval_accu {best_accu} at epoch {best_epoch}.")
    return results

In [ ]:
# Instantiate model_1
model_1 = BERTWeet_Classification(pretrained_model=bertweet_model).to(device)
print(f"bertweet trainable parameters: {trainable_parameters(bertweet_model)}")
print(f"model_1 trainable parameters: {trainable_parameters(model_1)}")                                  

# Define optimizer and loss function
optimizer_1 = torch.optim.AdamW(params=model_1.parameters(),
                               lr=0.001)   
loss_fn_1 = nn.CrossEntropyLoss()

# Train the model
results_1 = train_epoch(model=model_1, 
                        train_dataloader=train_dataloader, 
                        eval_dataloader=val_dataloader, 
                        optimizer=optimizer_1, 
                        loss_fn=loss_fn_1, 
                        early_stop_patience=5, 
                        checkpoint_file='best_model_1', 
                        device=device, 
                        epochs=20)

The first model, with 1538 trainable parameters, achieved a best validation accuracy around 0.8. I also tried unfreezing the pooler layer of the BERTWeet model, which increased the number of the trainable parameters to 592,130, but only improved the validation accuracy by about 1.6%. Therefore, I created another model by adding a LSTM layer on top of the last hidden state of the frozen BERTWeet model.

In [ ]:
class BERTweet_LSTM(nn.Module):
    def __init__(self, pretrained_model, lstm_layers=1, 
                 hidden_size=128, dropout=0.3):
        super().__init__()
        self.bertweet = pretrained_model
        self.dropout = nn.Dropout(p=dropout)
        self.norm = nn.LayerNorm(hidden_size)
        self.relu = nn.ReLU()

        self.lstm = nn.LSTM(input_size=768,
                            hidden_size=hidden_size,
                            num_layers=lstm_layers,
                            batch_first=True)

        self.classifier = nn.Linear(in_features=hidden_size,
                                    out_features=2)

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None):
        bert_outputs =  self.bertweet(input_ids=input_ids, 
                                      attention_mask=attention_mask, 
                                      token_type_ids=token_type_ids)

        # bert_outputs.last_hidden_state shape: [batch_size, seq_length, 768]
        # h_n shape: [num_layers, batch_size, hidden_size]
        _, (h_n, _) = self.lstm(bert_outputs.last_hidden_state)

        # Extract the final hidden state of the last LSTM layer
        # final_hidden shape: [batch_size, hidden_size]
        final_hidden = h_n[-1]

        # Normalization
        norm_output = self.norm(final_hidden)

        # Activation function
        relu_output = self.relu(norm_output)

        # Dropout
        dropout_output = self.dropout(relu_output)

        # Classifier
        logits = self.classifier(dropout_output)

        return logits

In [ ]:
# Instantiate model_2
model_2 = BERTweet_LSTM(pretrained_model=bertweet_model).to(device)
print(f"bertweet trainable parameters: {trainable_parameters(bertweet_model)}")
print(f"model_2 trainable parameters: {trainable_parameters(model_2)}")

# Define optimizer and loss function
optimizer_2 = torch.optim.AdamW(params=model_2.parameters(),
                               lr=0.0001)   
loss_fn_2 = nn.CrossEntropyLoss()

# Train the model
results_2 = train_epoch(model=model_2, 
                        train_dataloader=train_dataloader, 
                        eval_dataloader=val_dataloader, 
                        optimizer=optimizer_2, 
                        loss_fn=loss_fn_2, 
                        early_stop_patience=5, 
                        checkpoint_file='best_model_2', 
                        device=device, 
                        epochs=30)

Model_2 achieved an evaluation accuracy of 0.84. (***Note:** this value changes every time I run the notebook, even though I set the random seed at the beginning and in the Dataloader. I still haven't figuered out how to get completely reproducible results here...*) 

I then tuned the hyperparameter of this model using Ray Tune over 50 trials and got a set of best trial configuration, which I used to train the final model. (To save time, I omit the code here.)

In [ ]:
# Instantiate model_3
model_3 = BERTweet_LSTM(pretrained_model=bertweet_model,
                        lstm_layers=3, 
                        hidden_size=32, 
                        dropout=0.4).to(device)
print(f"bertweet trainable parameters: {trainable_parameters(bertweet_model)}")
print(f"model_3 trainable parameters: {trainable_parameters(model_3)}")

# Define optimizer and loss function
optimizer_3 = torch.optim.AdamW(params=model_3.parameters(),
                               lr=0.001, weight_decay=0.05)   
loss_fn_3 = nn.CrossEntropyLoss()

# Train the model
results_3 = train_epoch(model=model_3, 
                        train_dataloader=train_dataloader, 
                        eval_dataloader=val_dataloader, 
                        optimizer=optimizer_3, 
                        loss_fn=loss_fn_3, 
                        early_stop_patience=5, 
                        checkpoint_file='best_model_3', 
                        device=device, 
                        epochs=30)


It seems the evaluation accuracy of model_3 is a bit lower than that of model_2. Let's plot the history of both models.

In [ ]:
n = len(results_2['train_loss'])
model2_history = pd.DataFrame({
    'dataset': ['Train']*n + ['Validation']*n,
    'epoch': list(range(1, n+1))*2,
    'loss': results_2['train_loss'] + results_2['eval_loss'],
    'accuracy': results_2['train_accu'] + results_2['eval_accu']
})

m = len(results_3['train_loss'])
model3_history = pd.DataFrame({
    'dataset': ['Train']*m + ['Validation']*m,
    'epoch': list(range(1, m+1))*2,
    'loss': results_3['train_loss'] + results_3['eval_loss'],
    'accuracy': results_3['train_accu'] + results_3['eval_accu']
})

model2_history = model2_history.round({'loss': 4, 'accuracy': 4})
model3_history = model3_history.round({'loss': 4, 'accuracy': 4})

In [ ]:
import seaborn as sns
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
plt.subplots_adjust(hspace=0.3)

axes = axes.flatten()

sns.lineplot(data=model2_history, x='epoch', y='accuracy', hue='dataset', 
             palette=['skyblue', 'lightsalmon'], ax=axes[0])
axes[0].set_title('model_2: train_accu vs. eval_accu')
axes[0].legend(title='');

sns.lineplot(data=model3_history, x='epoch', y='accuracy', hue='dataset', 
             palette=['skyblue', 'lightsalmon'], ax=axes[1])
axes[1].set_title('model_3: train_accu vs. eval_accu')
axes[1].legend(title='');

sns.lineplot(data=model2_history, x='epoch', y='loss', hue='dataset', 
             palette=['skyblue', 'lightsalmon'], ax=axes[2])
axes[2].set_title('model_2: train_loss vs. eval_loss')
axes[2].legend(title='');

sns.lineplot(data=model3_history, x='epoch', y='loss', hue='dataset', 
             palette=['skyblue', 'lightsalmon'], ax=axes[3])
axes[3].set_title('model_3: train_loss vs. eval_loss')
axes[3].legend(title='');

From the the plots above, model_2 shows relatively smoother convergence, a more stable generalization gap, and more consistent improvements in both validation loss and accuracy. Model_3 was trained using the hyperparameter set from the best Ray Tune trial, but since I only searched 50 random configurations, it's possible that I didn't find the truly optimal setting.

I will use model_2 to make prediction on the test set.

In [ ]:
# Define the predict function
def predict(model, dataloader):
    model.eval()
    prediction = np.array([])
    with torch.inference_mode():
        for i, data in enumerate(dataloader):
            input_ids = data['input_ids']
            attention_mask = data['attention_mask']
            token_type_ids = data['token_type_ids']
            input_ids, attention_mask, token_type_ids = input_ids.to(device), attention_mask.to(device), token_type_ids.to(device)
            logits = model(input_ids, attention_mask, token_type_ids)
            y_pred = torch.argmax(logits, dim=-1).cpu().numpy()
            prediction = np.append(prediction, y_pred)
    return prediction.flatten().astype(int).tolist()

In [ ]:
# Make prediction on the test dataset
test_pred = predict(model_2, test_dataloader)

# Create submission dataframe
submission = pd.DataFrame({
    'id': clean_test['id'],
    'target': test_pred
})

submission.head()
submission.to_csv('submission.csv', index=False)

**Reference:** [BERTweet: A pre-trained language model for English Tweets](https://aclanthology.org/2020.emnlp-demos.2.pdf)